# Cohort properties

Self-contained derivation of the descriptive properties of the final 65-repository cohort (`scope-repository-cohort` v6.0.0), producing the two thesis tables `table:cohort` and `table:strata` plus the per-repository domain mapping. The review, discussion-rate and commit-attribution signals are reloaded here so the notebook runs independently of the sampling notebook.

## Setup

Installs the `cloc` binary and imports the libraries used for source-tree extraction and SLOC counting. Kept separate from the data cells because it requires network access and root, and runs once per environment.

In [1]:
import hashlib
import json
import subprocess
import tarfile
import tempfile
from pathlib import Path

import pygit2

# cloc drives the SLOC measure below; install it once per environment.
subprocess.run(["apt-get", "update", "-qq"], check=False, capture_output=True)
subprocess.run(
    ["apt-get", "install", "-y", "-qq", "cloc"], check=False, capture_output=True
)

CompletedProcess(args=['apt-get', 'install', '-y', '-qq', 'cloc'], returncode=0, stdout=b'Selecting previously unselected package perl-modules-5.40.\r\n(Reading database ... \r(Reading database ... 5%\r(Reading database ... 10%\r(Reading database ... 15%\r(Reading database ... 20%\r(Reading database ... 25%\r(Reading database ... 30%\r(Reading database ... 35%\r(Reading database ... 40%\r(Reading database ... 45%\r(Reading database ... 50%\r(Reading database ... 55%\r(Reading database ... 60%\r(Reading database ... 65%\r(Reading database ... 70%\r(Reading database ... 75%\r(Reading database ... 80%\r(Reading database ... 85%\r(Reading database ... 90%\r(Reading database ... 95%\r(Reading database ... 100%\r(Reading database ... 8030 files and directories currently installed.)\r\nPreparing to unpack .../00-perl-modules-5.40_5.40.1-6_all.deb ...\r\nUnpacking perl-modules-5.40 (5.40.1-6) ...\r\nSelecting previously unselected package libgdbm-compat4t64:arm64.\r\nPreparing to unpack .../01

## Data loading

Loads the assets the notebook derives its properties from: the 65-repository cohort, the issue catalog with review payloads, and the per-change discussion signal. Reloaded from storage so the notebook runs independently of the sampling notebook.

In [2]:
from benchmark_notebook import theme  # noqa: F401  registers thesis template
from benchmark_notebook.io import CloudStorage

import polars as pl

store = CloudStorage(bucket="bug-request-bucket")

In [3]:
cohort = pl.from_arrow(
    store.asset(
        key="scope-repository-cohort",
        version="6.0.0",
        location="main",
        partition="default",
    )
).select("name")
cohort.height

65

In [4]:
catalog = (
    pl.from_arrow(
        store.asset(
            key="scope-issue-catalog",
            version="4.0.0",
            location="main",
            partition="default",
        )
    )
    .filter(pl.col("type") == "change")
    .select(
        "name",
        "author",
        "number",
        "closed",
        pl.col("payload").struct.field("reviews").alias("reviews"),
    )
)

catalog.height

925617

In [5]:
discussion = pl.from_arrow(
    store.asset(
        key="scope-change-discussion",
        version="1.0.0",
        location="main",
        partition="default",
    )
)

discussed = (
    discussion.join(
        catalog.select("name", "number", "author", "closed").rename(
            {"author": "pr_author"}
        ),
        on=["name", "number"],
        how="inner",
    )
    .explode("comments")
    .unnest("comments")
    .filter(
        (pl.col("type") == "user")
        & (pl.col("author") != pl.col("pr_author"))
        & (pl.col("created") <= pl.col("closed"))
    )
    .group_by(["name", "number"])
    .agg(pl.len().alias("n_user_comments"))
    .with_columns((pl.col("n_user_comments") > 0).alias("discussed"))
    .join(catalog.select("name", "number"), on=["name", "number"], how="right")
    .with_columns(pl.col("discussed").fill_null(False))
    .select("name", "number", "discussed")
)

discussion_rate = (
    discussed.group_by("name")
    .agg(
        pl.len().alias("merged_prs"),
        pl.col("discussed").sum().alias("discussed_prs"),
    )
    .with_columns((pl.col("discussed_prs") / pl.col("merged_prs")).alias("share"))
    .join(cohort, on="name", how="inner")
    .sort("share")
)
discussion_rate.height

65

## Per-repository metrics

Assembles one row per repository from the loaded assets: review coverage and distinct reviewers, discussion-rate participation, core contributors at the Munaiah 80 % threshold, repository age, and Python SLOC counted over each repository's source tree. The result feeds both cohort tables.

In [6]:
# --- review: merged PRs, McIntosh peer-review coverage, distinct reviewers ---
reviews = (
    catalog.select("name", "author", "number", "reviews")
    .explode("reviews")
    .with_columns(
        pl.col("reviews").struct.field("type").alias("rtype"),
        pl.col("reviews").struct.field("author").alias("reviewer"),
    )
    .filter((pl.col("rtype") == "user") & (pl.col("reviewer") != pl.col("author")))
)

reviewers_per_month = (
    reviews.join(
        catalog.select("name", "number", "closed"), on=["name", "number"], how="left"
    )
    .with_columns(
        pl.col("closed")
        .str.to_datetime("%Y-%m-%dT%H:%M:%SZ", time_zone="UTC")
        .dt.truncate("1mo")
        .alias("month")
    )
    .group_by(["name", "month"])
    .agg(pl.col("reviewer").n_unique().alias("active_reviewers"))
    .group_by("name")
    .agg(pl.col("active_reviewers").mean().alias("reviewers_per_month"))
)

review = (
    catalog.select("name", "number")
    .join(
        reviews.select("name", "number")
        .unique()
        .with_columns(pl.lit(True).alias("peer")),
        on=["name", "number"],
        how="left",
    )
    .with_columns(pl.col("peer").fill_null(False))
    .group_by("name")
    .agg(
        pl.len().alias("merged_prs"),
        pl.col("peer").mean().alias("review_coverage"),
    )
    .join(
        reviews.group_by("name").agg(
            pl.col("reviewer").n_unique().alias("distinct_reviewers")
        ),
        on="name",
        how="left",
    )
    .with_columns(pl.col("distinct_reviewers").fill_null(0))
)

# --- participation = discussion rate, asset-faithful cut at [0.32, 0.48] ---
participation = discussion_rate.select("name", pl.col("share").alias("participation"))

# --- commit attribution: core contributors (Munaiah 80 %), repository age ---
attribution = (
    pl.from_arrow(
        store.asset(
            key="scope-commit-attribution",
            version="1.0.0",
            location="main",
            partition="default",
            columns=["name", "payload"],
        )
    )
    .join(cohort, on="name", how="inner")
    .with_columns(
        pl.col("payload").str.json_path_match("$.author.user.login").alias("login"),
        pl.col("payload")
        .str.json_path_match("$.committedDate")
        .str.to_datetime("%Y-%m-%dT%H:%M:%SZ", time_zone="UTC")
        .alias("committed"),
    )
)

age = attribution.group_by("name").agg(
    ((pl.col("committed").max() - pl.col("committed").min()).dt.total_days() / 30.44)
    .round(0)
    .cast(pl.Int64)
    .alias("age_months")
)

core = (
    attribution.filter(pl.col("login").is_not_null())
    .group_by("name", "login")
    .agg(pl.len().alias("commits"))
    .sort(["name", "commits"], descending=[False, True])
    .with_columns(
        (
            pl.col("commits").cum_sum().over("name")
            / pl.col("commits").sum().over("name")
        ).alias("covered"),
        pl.int_range(1, pl.len() + 1).over("name").alias("rank"),
    )
    .filter(pl.col("covered") >= 0.80)
    .group_by("name")
    .agg(pl.col("rank").min().alias("core_contributors"))
)

# --- SLOC: cloc Python source over each repository's source tarball ---
stationary = sorted(
    pl.from_arrow(
        store.asset(
            key="scope-repository-stationary",
            version="1.0.0",
            location="main",
            partition="default",
            columns=["name"],
        )
    )["name"].to_list()
)
ordered = sorted(repr(name) for name in stationary)
prefix = hashlib.sha256("".join(ordered).encode()).hexdigest()[:4]

blobs = {}
for blob in store._bucket.list_blobs(
    prefix="assets/scope-source-raw/version=4.0.0/location=main/"
):
    token = blob.name.split("partition=")[1].split("/")[0]
    if token[:4] != prefix:
        continue
    name = ordered[int(token[4:], 16)][1:-1]
    blobs.setdefault(name, blob.name)

# canonical/mysql-operator was cloned on a stray "readme" branch (its recorded trunk
# is wrong upstream); the real default branch is present as origin/main, so the
# tarball's HEAD is overridden for this one repository.
override = {"canonical/mysql-operator": "refs/remotes/origin/main"}

cache = Path(".jupyter/.cache/source")
cache.mkdir(parents=True, exist_ok=True)
gitlink = 0o160000
sloc = []
for name in sorted(cohort["name"].to_list()):
    archive = cache / (hashlib.sha256(name.encode()).hexdigest() + ".tar.gz")
    if not archive.exists():
        # download to a temporary sibling and rename on success, so an interrupted
        # download never leaves a truncated archive that later reads as valid
        partial = archive.with_suffix(".tar.gz.partial")
        store._bucket.blob(blobs[name]).download_to_filename(str(partial))
        partial.replace(archive)
    with tempfile.TemporaryDirectory() as work:
        with tarfile.open(archive, "r:gz") as tar:
            tar.extractall(work, filter="data")
        git = pygit2.Repository(str(next(Path(work).rglob("repository.git"))))
        head = git.lookup_reference(override[name]) if name in override else git.head
        index = pygit2.Index()
        index.read_tree(head.peel(pygit2.Tree))
        tree = Path(work) / "tree"
        for entry in index:
            if entry.mode == gitlink or entry.id not in git:
                continue
            destination = tree / entry.path
            destination.parent.mkdir(parents=True, exist_ok=True)
            destination.write_bytes(git[entry.id].data)
        report = json.loads(
            subprocess.run(
                ["cloc", "--json", "--quiet", str(tree)],
                capture_output=True,
                text=True,
            ).stdout
            or "{}"
        )
        sloc.append({"name": name, "sloc": report.get("Python", {}).get("code", 0)})

sloc = pl.DataFrame(sloc)

# --- assemble per-repo table ---
repo = (
    review.join(participation, on="name", how="inner")
    .join(core, on="name", how="inner")
    .join(age, on="name", how="inner")
    .join(sloc, on="name", how="inner")
    .join(reviewers_per_month, on="name", how="inner")
    .select(
        "name",
        "merged_prs",
        pl.col("review_coverage").round(2),
        "distinct_reviewers",
        pl.col("reviewers_per_month").round(2),
        pl.col("participation").round(2),
        "core_contributors",
        "age_months",
        "sloc",
    )
    .with_columns((pl.col("merged_prs") / pl.col("age_months")).alias("pr_rate"))
)
repo.height

65

## Cohort tables

Produces the two thesis tables. `table:cohort` reports min, median and max per metric across the 65 repositories; `table:strata` reports the repository count per discussion-rate stratum, cut at 0.32 and 0.48.

In [7]:
# table:cohort — min / median / max per metric across the 65 repositories
metrics = [
    "merged_prs",
    "review_coverage",
    "distinct_reviewers",
    "participation",
    "core_contributors",
    "age_months",
    "sloc",
    "pr_rate",
    "reviewers_per_month",
]

cohort_table = (
    repo.unpivot(on=metrics, variable_name="metric", value_name="value")
    .group_by("metric")
    .agg(
        pl.col("value").min().alias("min"),
        pl.col("value").median().alias("median"),
        pl.col("value").max().alias("max"),
    )
    .with_columns(pl.col("metric").cast(pl.Enum(metrics)))
    .sort("metric")
)

with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=40, float_precision=2):
    print(cohort_table)

shape: (9, 4)
┌─────────────────────┬─────────┬──────────┬───────────┐
│ metric              ┆ min     ┆ median   ┆ max       │
│ ---                 ┆ ---     ┆ ---      ┆ ---       │
│ enum                ┆ f64     ┆ f64      ┆ f64       │
╞═════════════════════╪═════════╪══════════╪═══════════╡
│ merged_prs          ┆ 124.00  ┆ 343.00   ┆ 3834.00   │
│ review_coverage     ┆ 0.89    ┆ 1.00     ┆ 1.00      │
│ distinct_reviewers  ┆ 4.00    ┆ 24.00    ┆ 106.00    │
│ participation       ┆ 0.11    ┆ 0.44     ┆ 0.91      │
│ core_contributors   ┆ 1.00    ┆ 6.00     ┆ 46.00     │
│ age_months          ┆ 20.00   ┆ 37.00    ┆ 115.00    │
│ sloc                ┆ 3819.00 ┆ 19070.00 ┆ 218852.00 │
│ pr_rate             ┆ 2.86    ┆ 8.16     ┆ 49.15     │
│ reviewers_per_month ┆ 1.81    ┆ 4.81     ┆ 16.41     │
└─────────────────────┴─────────┴──────────┴───────────┘


In [8]:
# table:strata — repositories per discussion-rate stratum (cut at [0.32, 0.48])
# Stratum is assigned on the full-precision discussion rate, identical to the
# scope_repository_sample asset; the participation span is rounded only for display.
strata_table = (
    discussion_rate.select("name", "share")
    .with_columns(
        pl.col("share")
        .cut(breaks=[0.32, 0.48], labels=["low", "mid", "high"])
        .alias("stratum")
    )
    .group_by("stratum")
    .agg(
        pl.col("share").min().round(2).alias("participation_min"),
        pl.col("share").max().round(2).alias("participation_max"),
        pl.len().alias("count"),
    )
    .with_columns(pl.col("stratum").cast(pl.Enum(["low", "mid", "high"])))
    .sort("stratum")
)

assert strata_table["count"].sum() == 65

with pl.Config(tbl_rows=-1, tbl_cols=-1, float_precision=2):
    print(strata_table)

shape: (3, 4)
┌─────────┬───────────────────┬───────────────────┬───────┐
│ stratum ┆ participation_min ┆ participation_max ┆ count │
│ ---     ┆ ---               ┆ ---               ┆ ---   │
│ enum    ┆ f64               ┆ f64               ┆ u32   │
╞═════════╪═══════════════════╪═══════════════════╪═══════╡
│ low     ┆ 0.11              ┆ 0.31              ┆ 17    │
│ mid     ┆ 0.33              ┆ 0.48              ┆ 23    │
│ high    ┆ 0.48              ┆ 0.91              ┆ 25    │
└─────────┴───────────────────┴───────────────────┴───────┘


## Figures

The distribution of the discussion rate across the cohort, and the activity-versus-scale bubble figure shaded by discussion rate.

In [20]:
import plotly.express as px
import plotly.io as pio

px.histogram(
    x=discussion_rate["share"], 
    color_discrete_sequence=[pio.templates["thesis"].layout.colorway[3]]
)

In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

size = repo["age_months"].to_numpy()
color = repo["participation"].to_numpy()
sizeref = 10.0 * max(size) / (24.0**2)
names = repo["name"].to_list()


def bubbles(x, y):
    return go.Scatter(
        x=x,
        y=y,
        mode="markers",
        text=names,
        showlegend=False,
        marker=dict(
            size=5,
            color=color,
            coloraxis="coloraxis",
            line=dict(width=0.0),
            opacity=0.85,
        ),
        hovertemplate="%{text}<br>%{x}, %{y}<extra></extra>",
    )


fig = make_subplots(
    rows=1,
    cols=2,
    horizontal_spacing=0.12,
)

fig.add_trace(
    bubbles(repo["pr_rate"].to_numpy(), repo["reviewers_per_month"].to_numpy()),
    row=1,
    col=1,
)
fig.add_trace(
    bubbles(repo["sloc"].to_numpy(), repo["core_contributors"].to_numpy()),
    row=1,
    col=2,
)

fig.update_xaxes(title="Merges per Month", row=1, col=1)
fig.update_yaxes(title="Reviewers per Month", row=1, col=1)
fig.update_xaxes(type="log", title="Lines of Code", row=1, col=2)
fig.update_yaxes(title="Core Contributors", row=1, col=2)


fig.update_layout(
    template="thesis",
    width=800,
    height=400,
    coloraxis=dict(
        colorbar=dict(
            title="Discussion Rate",
            thickness=12,
            len=0.8,
        ),
    ),
)
fig.show(
    config={
        "toImageButtonOptions": {
            "format": "svg",
            "filename": "cohort_funnel",
            "height": 400,
            "width": 800,
            "scale": 1,
        }
    }
)

## Domain classification

GitHub topics are owner-assigned and sparse (26 of 65 repositories carry any topic), so the application domain is not directly available. Each repository is instead mapped to the closest entry in a fixed, citable taxonomy of software-engineering topics — the Wikidata-grounded ranking of \textcite{disipio-2021} (301 topics, each with a label and gloss). A language model selects the single best-matching topic from this catalogue using the repository name, description, topics and primary language. The taxonomy is the authoritative label set; the model only assigns. The resulting per-repository mapping feeds the manual domain buckets and is not aggregated here.

In [11]:
import os
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path

from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field
from tqdm import tqdm

taxonomy = pl.read_csv(
    "https://zenodo.org/records/5879573/files/mean_std_ranking.csv?download=1"
)
catalogue = "\n".join(
    f"{row['q_id']}\t{row['topic']}\t{row['description']}"
    for row in taxonomy.iter_rows(named=True)
)

meta = pl.from_arrow(
    store.asset(
        key="scope-repository-cohort",
        version="6.0.0",
        location="main",
        partition="default",
        columns=["name", "description", "topics", "language"],
    )
)


class Domain(BaseModel):
    q_id: str = Field(
        description="Wikidata Q-ID of the single best-matching taxonomy topic"
    )


classifier = ChatOpenAI(
    model="Qwen/Qwen3.7-Max",
    api_key=os.getenv("TOGETHER_AI_API_KEY"),
    base_url="https://api.together.xyz/v1",
    temperature=0.0,
    max_tokens=4096,
    streaming=True,
).with_structured_output(Domain)

instruction = (
    "You assign a software repository to the single best-matching topic from a fixed "
    "taxonomy of software-engineering domains. Choose the topic that best captures the "
    "repository's application domain. Return a JSON object with its exact Wikidata Q-ID.\n\n"
    "TAXONOMY (q_id<tab>topic<tab>description):\n" + catalogue
)


def classify(row):
    try:
        return classifier.invoke(
            f"{instruction}\n\nREPOSITORY\nname: {row['name']}\n"
            f"language: {row['language']}\ntopics: {row['topics']}\n"
            f"description: {row['description']}"
        ).q_id
    except Exception:
        return None


# Parallel over repositories with a progress bar; incremental and resumable. Each result
# is written to domain.parquet from the main thread as soon as its future completes, so a
# crash loses nothing and a re-run skips repositories already classified (no duplicate
# calls). A failed call yields a null q_id rather than aborting, and is retried next run
# because null rows are not counted as done.
store_path = Path("domain.parquet")
done = (
    pl.read_parquet(store_path)
    if store_path.exists()
    else pl.DataFrame(schema={"name": pl.Utf8, "q_id": pl.Utf8})
)
classified = {
    row["name"]: row["q_id"]
    for row in done.iter_rows(named=True)
    if row["q_id"] is not None
}
pending = [row for row in meta.iter_rows(named=True) if row["name"] not in classified]

with ThreadPoolExecutor(max_workers=8) as pool:
    futures = {pool.submit(classify, row): row["name"] for row in pending}
    for future in tqdm(as_completed(futures), total=len(futures), desc="domains"):
        classified[futures[future]] = future.result()
        pl.DataFrame(
            {"name": list(classified), "q_id": list(classified.values())}
        ).write_parquet(store_path)

domain = pl.read_parquet(store_path).join(
    taxonomy.select("q_id", "topic"), on="q_id", how="left"
)
unmatched = domain.filter(pl.col("topic").is_null())
print(
    f"{domain.height} repositories classified, {unmatched.height} unmatched or failed"
)

with pl.Config(tbl_rows=-1, tbl_cols=-1, fmt_str_lengths=60):
    print(domain.sort("topic"))

domains: 0it [00:00, ?it/s]

65 repositories classified, 0 unmatched or failed
shape: (65, 3)
┌──────────────────────────────────────────────────────┬────────────┬─────────────────────────────┐
│ name                                                 ┆ q_id       ┆ topic                       │
│ ---                                                  ┆ ---        ┆ ---                         │
│ str                                                  ┆ str        ┆ str                         │
╞══════════════════════════════════════════════════════╪════════════╪═════════════════════════════╡
│ canonical/github-runner-operator                     ┆ Q3025536   ┆ DevOps                      │
│ thunderbird/pulumi                                   ┆ Q3025536   ┆ DevOps                      │
│ astronomy-commons/hats-import                        ┆ Q333       ┆ astronomy                   │
│ stytchauth/stytch-python                             ┆ Q212108    ┆ authentication              │
│ datarobot/airflow-provider-dataro

## Per-pull-request metrics

Four pull-request-level metrics over the merged pull requests of the 65 cohort repositories (`scope-issue-catalog` v4.0.0, `type == "change"`, `payload.merged` not null). Diff size and changed-file counts are restricted to Python source files, taken from the per-file paths in `scope-change-outline` v3.0.0, so generated, vendored and lock files (which otherwise inflate the diff into the millions of lines) are excluded — consistent with the Python-only size measure above. The population is therefore the merged pull requests that touch at least one Python file; one row per pull request, then aggregated. Label presence comes from the catalog `labels`. A linked-issue signal is not available from a complete source — `scope-reference-edge` covers under one percent of pull requests — so `has_issue` is reported as missing rather than from an incomplete asset.

In [12]:
# Merged change rows of the cohort, with label presence (one row per pull request).
merged = (
    pl.from_arrow(
        store.asset(
            key="scope-issue-catalog",
            version="4.0.0",
            location="main",
            partition="default",
        )
    )
    .filter(pl.col("type") == "change")
    .join(cohort, on="name", how="inner")
    .filter(pl.col("payload").struct.field("merged").is_not_null())
    .select("name", "number", (pl.col("labels").list.len() > 0).alias("has_label"))
)

# Diff size and changed-file count restricted to Python source files, from the outline.
# This excludes generated, vendored and lock files (e.g. JSON fixtures) that otherwise
# dominate the diff size, consistent with the Python-only SLOC measure above.
python = (
    pl.from_arrow(
        store.asset(
            key="scope-change-outline",
            version="3.0.0",
            location="main",
            partition="default",
            columns=["name", "number", "files"],
        )
    )
    .join(cohort, on="name", how="inner")
    .explode("files")
    .with_columns(
        pl.col("files").struct.field("path").alias("path"),
        (
            pl.col("files").struct.field("additions")
            + pl.col("files").struct.field("deletions")
        ).alias("changed"),
    )
    .filter(pl.col("path").str.ends_with(".py"))
    .group_by("name", "number")
    .agg(
        pl.col("changed").sum().alias("lines"),
        pl.len().alias("files"),
    )
)

# Population: merged pull requests that touch at least one Python file.
pr = merged.join(python, on=["name", "number"], how="inner")

per_pr = pl.DataFrame(
    {
        "metric": ["lines per PR", "files per PR"],
        "min": [pr["lines"].min(), pr["files"].min()],
        "median": [int(pr["lines"].median()), int(pr["files"].median())],
        "max": [pr["lines"].max(), pr["files"].max()],
    }
)

shares = pl.DataFrame(
    {
        "metric": ["has_label", "has_issue"],
        "share": [round(pr["has_label"].mean(), 2), None],
    }
)

print(f"pull requests touching Python: {pr.height}")
with pl.Config(tbl_rows=-1, tbl_cols=-1):
    print(per_pr)
    print(shares)
print(
    "has_issue: no complete source — scope-reference-edge covers <1% of PRs; "
    "full linked-issue recovery needs scope-change-raw timeline events."
)

pull requests touching Python: 29461
shape: (2, 4)
┌──────────────┬─────┬────────┬────────┐
│ metric       ┆ min ┆ median ┆ max    │
│ ---          ┆ --- ┆ ---    ┆ ---    │
│ str          ┆ i64 ┆ i64    ┆ i64    │
╞══════════════╪═════╪════════╪════════╡
│ lines per PR ┆ 0   ┆ 49     ┆ 164834 │
│ files per PR ┆ 1   ┆ 2      ┆ 602    │
└──────────────┴─────┴────────┴────────┘
shape: (2, 2)
┌───────────┬───────┐
│ metric    ┆ share │
│ ---       ┆ ---   │
│ str       ┆ f64   │
╞═══════════╪═══════╡
│ has_label ┆ 0.26  │
│ has_issue ┆ null  │
└───────────┴───────┘
has_issue: no complete source — scope-reference-edge covers <1% of PRs; full linked-issue recovery needs scope-change-raw timeline events.
